<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f172a 0%, #7c3aed 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.25);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 FLUX 3 Image Klein</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition · <strong>experimental</strong></h3>
  <p style='color: #e5e5e5; margin: 0;'>5.6B open-weights image model by Black Forest Labs | text-to-image | editing | up to 10 reference images | FP16 split over both T4s</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <img src="https://img.shields.io/badge/Model-flux--3--image--klein-7c3aed?style=for-the-badge&logo=huggingface&logoColor=white" />
  <img src="https://img.shields.io/badge/Status-experimental-yellow?style=for-the-badge" />
</div>

---

### What is FLUX 3 Image Klein?

Released by Black Forest Labs on 15 October 2026: a **5.6B** step-distilled image model (16 steps, no CFG) that does **text-to-image, single-image editing and multi-reference composition (up to 10 images)** with one checkpoint, renders text, and understands **bounding boxes** written in the prompt. Its text encoder is Qwen3-VL-4B-Instruct.

### How does it run on two T4s?

The official builds are BF16, FP8 and NVFP4, none of which the T4 (Turing, sm75) runs natively. This notebook uses the BF16 weights converted to **FP16** at load time, through Hugging Face Diffusers (pinned commit):

- **GPU 0:** the 5.6B transformer (~10.5 GB) · **GPU 1:** the Qwen3-VL-4B text encoder (~8.4 GB) + the VAE in FP32
- **FP16 overflow guards.** The model was trained in BF16, whose range FP16 lacks: plain FP16 gives **black images**. The residual stream is kept in FP32 (it grows past 1 million), the 20480→3072 text projection runs in FP32 (Qwen3-VL's outlier channels reach ~11k), and the feed-forward and attention-output matmuls are rescaled when they would exceed 65504. Every matmul still runs in FP16 on the tensor cores.

### Measured on Kaggle T4 x2

| | |
|---|---|
| Download + install | ~2-3 min (~20 GB) |
| Load | ~1 min |
| 1024×1024 or 768×1344, 16 steps | ~50-70 s |
| Edit with one 1024 px reference | ~2-2.5 min |
| VRAM | ~11.8 GiB on GPU 0, ~11.6 GiB on GPU 1 |

### ⚠️ Before you start
1. **Accelerator:** GPU T4 x2 · **Internet:** on
2. **Weights:** the official repo is gated. If you add a Kaggle secret `HF_TOKEN` (Add-ons → Secrets) from an account that accepted the license on [the model page](https://huggingface.co/black-forest-labs/flux-3-image-klein), Step 1 downloads from it; otherwise it uses the ungated copy `ml-resources/flux-3-image-klein`, whose weight files have the same SHA-256 as the official ones.
3. **License:** [FLUX Kommunity License](https://huggingface.co/black-forest-labs/flux-3-image-klein/blob/main/LICENSE.md): non-commercial use, plus commercial use of outputs for individuals and companies under USD 5M annual revenue. Read Section 4 for the forbidden uses. Text encoder: Apache-2.0.

---
## ⚙️ Step 1: Setup + model download

Installs Diffusers (pinned commit) and recent Transformers, then downloads the transformer, the text encoder and the VAE (~20 GB) to `/kaggle/tmp`. An interrupted download resumes on its own.

In [ ]:
# Step 1: dependencies + weights (~20 GB)
"""FLUX 3 Image Klein (distilled, 5.6B) on 2x T4: shared by the notebook and the test kernel.

T4 (sm75) has no BF16/FP8 tensor cores, so everything runs in FP16:
  GPU 0: the 5.6B transformer (11.1 GB fp16)
  GPU 1: the Qwen3-VL-4B text encoder (8.9 GB fp16) + the VAE
"""
import gc
import json
import os
import subprocess
import sys
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

DIFFUSERS_COMMIT = "7ecb1e1b4fe500be7fb7381a7bb79a24d7fa3941"  # huggingface/diffusers main, 15 Oct 2026
OFFICIAL_REPO = "black-forest-labs/flux-3-image-klein"  # gated (accept the license on HF + HF_TOKEN)
MIRROR_REPO = "ml-resources/flux-3-image-klein"  # ungated copy, identical LFS sha256 on every weight file
ROOT = "/kaggle/tmp" if os.path.isdir("/kaggle") else os.path.abspath(".")
CKPT_DIR = os.path.join(ROOT, "flux-3-image-klein")
PATTERNS = ["modular_model_index.json", "transformer/*", "vae/*", "text_encoder/*", "LICENSE.md", "NOTICE.md"]


def sh(cmd):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, shell=isinstance(cmd, str))


def setup():
    t0 = time.time()
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception:
        pass
    sh([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5.0", "accelerate", "huggingface_hub", "hf_xet",
        "safetensors", "gradio>=5.0", f"git+https://github.com/huggingface/diffusers@{DIFFUSERS_COMMIT}"])
    # Kaggle ships an old torchao that breaks diffusers' import; this notebook doesn't use it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    from huggingface_hub import snapshot_download
    repo = MIRROR_REPO
    if os.environ.get("HF_TOKEN"):
        try:
            from huggingface_hub import hf_hub_download
            hf_hub_download(OFFICIAL_REPO, "modular_model_index.json", local_dir=CKPT_DIR)
            repo = OFFICIAL_REPO
        except Exception as e:
            print("Official repo not accessible with this HF_TOKEN (accept the license on its page). Using the mirror.", repr(e)[:200])
    print(f"⬇️ Downloading {repo} (~20 GB)...", flush=True)
    for attempt in range(5):
        try:
            snapshot_download(repo, local_dir=CKPT_DIR, allow_patterns=PATTERNS, max_workers=8)
            break
        except Exception as e:
            print("download retry", attempt, repr(e)[:300], flush=True)
            time.sleep(20)
    # Point every component at the local folder instead of the gated hub repo.
    idx = os.path.join(CKPT_DIR, "modular_model_index.json")
    s = open(idx).read().replace(f'"{OFFICIAL_REPO}"', json.dumps(CKPT_DIR))
    open(idx, "w").write(s)
    print(f"✅ Setup done in {time.time() - t0:.0f}s", flush=True)


def gpu_report():
    import torch
    return " · ".join(f"GPU{i} {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GiB peak" for i in range(torch.cuda.device_count()))


def load_pipeline():
    import torch
    from diffusers import ModularPipeline

    t0 = time.time()
    pipe = ModularPipeline.from_pretrained(CKPT_DIR)
    pipe.load_components(dtype={"default": torch.float16, "vae": torch.float32},
                         device_map={"transformer": "cuda:0", "text_encoder": "cuda:1", "vae": "cuda:1"})
    # Latents live on GPU 0 next to the transformer.
    type(pipe)._execution_device = property(lambda self: torch.device("cuda:0"))
    te, vae = pipe.text_encoder, pipe.vae
    te.to("cuda:1"); vae.to("cuda:1"); pipe.transformer.to("cuda:0")

    # Text encoder: inputs arrive on GPU 0, run on GPU 1 (outputs are moved back by the pipeline).
    def te_pre(mod, args, kwargs):
        return (tuple(a.to("cuda:1") if torch.is_tensor(a) else a for a in args),
                {k: v.to("cuda:1") if torch.is_tensor(v) else v for k, v in kwargs.items()})
    te.register_forward_pre_hook(te_pre, with_kwargs=True)

    # VAE (FP32, as its config asks) on GPU 1, results back to GPU 0.
    orig_dec, orig_enc = vae.decode, vae.encode

    def decode(z, *a, **k):
        out = orig_dec(z.to("cuda:1"), *a, **k)
        if isinstance(out, tuple):
            return tuple(o.to("cuda:0") if torch.is_tensor(o) else o for o in out)
        out.sample = out.sample.to("cuda:0")
        return out

    def encode(x, *a, **k):
        out = orig_enc(x.to("cuda:1"), *a, **k)
        d = out.latent_dist if hasattr(out, "latent_dist") else out[0]
        for name in ("parameters", "mean", "logvar", "std", "var"):
            if torch.is_tensor(getattr(d, name, None)):
                setattr(d, name, getattr(d, name).to("cuda:0"))
        return out
    vae.decode, vae.encode = decode, encode
    apply_fp16_fixes(pipe)
    try:
        vae.enable_tiling()
    except Exception:
        pass
    gc.collect(); torch.cuda.empty_cache()
    print(f"📦 Loaded in {time.time() - t0:.0f}s · " + " · ".join(
        f"GPU{i} {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB" for i in range(torch.cuda.device_count())), flush=True)
    return pipe


def generate(pipe, prompt, images=None, width=None, height=None, steps=None, seed=-1):
    import torch
    seed = int(seed) if seed is not None and int(seed) >= 0 else int(torch.randint(0, 2**31 - 1, (1,)))
    kw = dict(prompt=prompt, generator=torch.Generator().manual_seed(seed), output="images")
    if images:
        kw["image"] = images if len(images) > 1 else images[0]
    if width and height:
        kw["width"], kw["height"] = int(width) // 16 * 16, int(height) // 16 * 16
    if steps:
        kw["num_inference_steps"] = int(steps)
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(**kw)[0]
    took = time.time() - t0
    print(f"🖼️ {img.size[0]}x{img.size[1]} in {took:.1f}s (seed {seed}) · {gpu_report()}", flush=True)
    return img, took, seed


# ---------------------------------------------------------------------------------------------------------------
# FP16 safety for Turing: the model was trained in BF16, whose range FP16 does not have. The transformer's residual
# stream (and the SwiGLU product) is kept in FP32 while every matmul stays in FP16 on the tensor cores.
# ---------------------------------------------------------------------------------------------------------------
def _safe_linear(lin, x, s=1.0):
    """FP16 matmul whose output may exceed 65504: run it on x/s, retry with a bigger s if anything overflowed,
    and return the rescaled result in FP32. Costs one sync per call."""
    import torch
    for _ in range(4):
        y = lin(x if s == 1.0 else x * (1.0 / s))
        if bool(torch.isfinite(y).all()):
            return y.float() * s if s != 1.0 else y.float()
        s *= 64.0
    return y.float() * s


def _ff_forward(self, x):
    import torch
    h = _safe_linear(self.linear_in, x)
    half = h.shape[-1] // 2
    a = torch.nn.functional.silu(h[..., :half]) * h[..., half:]
    del h
    s = float((a.abs().amax() / 32768.0).clamp(min=1.0))
    return _safe_linear(self.linear_out, (a / s).to(x.dtype), 1.0) * s


def _mode_forward(self, hidden_states, temb_mod, image_rotary_emb):
    import torch
    dt = self.ff.linear_in.weight.dtype
    shift, scale, gate = (t.float() for t in torch.chunk(temb_mod, 3, dim=-1))
    h = hidden_states.float()
    n = ((1 + scale) * self.norm(h) + shift).to(dt)
    out = self.attn(hidden_states=n, image_rotary_emb=image_rotary_emb).float() + self.ff(n)
    return h + gate * out


def _single_forward(self, hidden_states, encoder_hidden_states, temb_mod, temb_mod_context, image_rotary_emb):
    import torch
    dt = self.ff.linear_in.weight.dtype
    shift, scale, gate = (t.float() for t in torch.chunk(temb_mod, 3, dim=-1))
    c_shift, c_scale, c_gate = (t.float() for t in torch.chunk(temb_mod_context, 3, dim=-1))
    h, e = hidden_states.float(), encoder_hidden_states.float()
    n = (1 + scale) * self.norm(h) + shift
    ne = (1 + c_scale) * self.norm(e) + c_shift
    combined = torch.cat([ne, n], dim=1).to(dt)
    out = self.attn(hidden_states=combined, image_rotary_emb=image_rotary_emb).float() + self.ff(combined)
    co, ho = torch.split(out, [e.shape[1], h.shape[1]], dim=1)
    return e + c_gate * co, h + gate * ho


def apply_fp16_fixes(pipe):
    import types
    import torch
    tr = pipe.transformer
    for m in tr.modules():
        name = type(m).__name__
        if name == "Flux3FeedForward":
            m.forward = types.MethodType(_ff_forward, m)
        elif name == "Flux3ModeTransformerBlock":
            m.forward = types.MethodType(_mode_forward, m)
        elif name == "Flux3SingleTransformerBlock":
            m.forward = types.MethodType(_single_forward, m)
        elif name == "Flux3Attention":
            lin = m.to_out[0]
            lin.forward = types.MethodType(lambda self, x, _f=lin.forward: _safe_linear(_f, x), lin)
    # Output layers receive the FP32 stream.
    def cast_in(mod, args, kwargs):
        dt = next(mod.parameters()).dtype
        return tuple(a.to(dt) if torch.is_tensor(a) and a.is_floating_point() else a for a in args), kwargs
    tr.proj_out.register_forward_pre_hook(cast_in, with_kwargs=True)
    # Qwen3-VL's hidden states carry outlier channels (|x| ~ 11k); the 20480 -> 3072 text projection overflows FP16
    # on them, so it runs in FP32 (250 MB) and feeds the FP32 text stream.
    tr.context_embedder.float()
    tr.context_embedder.register_forward_pre_hook(cast_in, with_kwargs=True)
    # The image cond tokens (fp16) are concatenated with the fp32 image stream: cast them up.
    return pipe


setup()

---
## 📦 Step 2: Load the model on both GPUs

About 1 minute. Re-run only if you restarted the session.

In [ ]:
# Step 2: load FLUX 3 Image Klein (transformer on GPU 0, text encoder + VAE on GPU 1)
import torch
try:
    pipe
except NameError:
    pipe = load_pipeline()

---
## 🚀 Step 3: Gradio UI

Prints a public **gradio.live** link. Two tabs: **Text → Image**, and **Edit / References** (1 to 10 images). Generations run one at a time.

In [ ]:
# Step 3: Gradio interface
import os, threading, time
import gradio as gr
from PIL import Image

LOCK = threading.Lock()
SIZES = {"1024 × 1024 (1:1)": (1024, 1024), "1152 × 896 (4:3)": (1152, 896), "896 × 1152 (3:4)": (896, 1152),
         "1344 × 768 (16:9)": (1344, 768), "768 × 1344 (9:16)": (768, 1344), "1536 × 640 (21:9)": (1536, 640),
         "768 × 768 (fast)": (768, 768)}
OUT = "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)

def _save(img):
    path = os.path.join(OUT, time.strftime("%Y%m%d-%H%M%S") + ".png")
    img.save(path)
    return path

def t2i(prompt, size, steps, seed):
    if not prompt.strip():
        raise gr.Error("Write a prompt first.")
    w, h = SIZES[size]
    with LOCK:
        img, took, seed = generate(pipe, prompt, None, w, h, steps, seed)
    _save(img)
    return img, f"{w}×{h} · {int(steps)} steps · seed {seed} · {took:.0f}s · {gpu_report()}"

def edit(files, prompt, size, steps, seed):
    if not files:
        raise gr.Error("Add at least one reference image.")
    if not prompt.strip():
        raise gr.Error("Describe the image you want.")
    refs = [Image.open(f if isinstance(f, str) else f.name).convert("RGB") for f in files][:10]
    for r in refs:
        r.thumbnail((1024, 1024))
    wh = SIZES.get(size) if size != "Same as first reference" else (None, None)
    with LOCK:
        img, took, seed = generate(pipe, prompt, refs, wh[0], wh[1], steps, seed)
    _save(img)
    return img, f"{img.size[0]}×{img.size[1]} · {len(refs)} ref · {int(steps)} steps · seed {seed} · {took:.0f}s · {gpu_report()}"

with gr.Blocks(title="FLUX 3 Image Klein · T4 x2") as demo:
    gr.Markdown("## 🎨 FLUX 3 Image Klein · Kaggle T4 x2\nWrite **full scene captions** (subject, framing, background, light): short prompts give generic images.")
    with gr.Tab("Text → Image"):
        with gr.Row():
            with gr.Column():
                p1 = gr.Textbox(label="Prompt", lines=5, value="A hand-painted wooden sign hanging outside a small bakery in Dakar, the sign reads 'BOULANGERIE TERANGA' in bold white letters on a deep blue background, warm late-afternoon sunlight, shallow depth of field, photorealistic.")
                s1 = gr.Dropdown(list(SIZES), value="1024 × 1024 (1:1)", label="Size")
                with gr.Row():
                    st1 = gr.Slider(4, 32, value=16, step=1, label="Steps (16 = model default)")
                    sd1 = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
                b1 = gr.Button("Generate", variant="primary")
            with gr.Column():
                o1 = gr.Image(label="Result", format="png")
                i1 = gr.Markdown()
        b1.click(t2i, [p1, s1, st1, sd1], [o1, i1], api_name="t2i")
    with gr.Tab("Edit / References"):
        with gr.Row():
            with gr.Column():
                f2 = gr.File(file_count="multiple", file_types=["image"], label="Reference images (1-10)")
                p2 = gr.Textbox(label="Target caption: describe the whole result image, edit included", lines=5)
                s2 = gr.Dropdown(["Same as first reference"] + list(SIZES), value="Same as first reference", label="Size")
                with gr.Row():
                    st2 = gr.Slider(4, 32, value=16, step=1, label="Steps")
                    sd2 = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
                b2 = gr.Button("Generate", variant="primary")
            with gr.Column():
                o2 = gr.Image(label="Result", format="png")
                i2 = gr.Markdown()
        b2.click(edit, [f2, p2, s2, st2, sd2], [o2, i2], api_name="edit")

# "Save & Run All" (batch) sessions have nobody to click: run one test generation, then stop.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    img, info = t2i("A wide-angle photograph of a lone American bison in a misty geothermal landscape, half hidden by white steam, a slate-blue stream and evergreens fading into the mist behind. Muted, cool palette, soft overcast light.", "1024 × 1024 (1:1)", 16, 42)
    img.save("/kaggle/working/batch_test.png")
    print("🧪 Batch test:", info)
    path = "/kaggle/working/batch_test.png"
    img2, info2 = edit([path], "The same lone American bison in the same misty geothermal landscape, now at night under a full moon: the steam lit from behind, cool blue light, deep shadows, stars above the evergreens.", "Same as first reference", 16, 42)
    img2.save("/kaggle/working/batch_edit.png")
    print("🧪 Batch edit:", info2)
else:
    demo.queue(default_concurrency_limit=1).launch(share=True, debug=False, inline=False)

---
## ✍️ Prompting tips

- **Full captions win.** Describe framing, subject, background and lighting. For edits, describe the *whole target image* with the change folded in, not just "make it blue".
- **Text in images:** put the exact words in quotes.
- **Layout with boxes:** append a JSON list after the caption, boxes as `[top, left, bottom, right]` on a 0-1000 grid, and tag elements in the caption with `<id>`:
  `... a black silhouette <silhouette_1> on a chartreuse background <background_1>. [{"id": "background_1", "bbox": [0, 0, 1000, 1000], "desc": "..."}, {"id": "silhouette_1", "bbox": [150, 150, 850, 850], "desc": "..."}]`
- **References:** each image keeps its own resolution (resized to ≤1024 px here); more references make generation slower.
- BFL's 18.8 GB prompt upsampler is not included: it does not fit next to the model on T4s.

**Troubleshooting**
- `CUDA out of memory`: pick a smaller size or fewer references, then retry.
- Black image: please report it with the prompt and size; the FP16 guards should prevent it.

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  Model: <a href="https://huggingface.co/black-forest-labs/flux-3-image-klein">black-forest-labs/flux-3-image-klein</a> · Code: <a href="https://github.com/huggingface/diffusers">huggingface/diffusers</a> · FLUX Kommunity License
</p>